# Silver Layer Pipeline

**Medallion Architecture - Silver Layer**

This notebook transforms raw Bronze layer data into clean, standardized Silver layer tables optimized for analytics.

**Pipeline Stages:**
1. **Configuration** - Environment setup and path configuration
2. **Data Discovery** - Identify Bronze layer tables
3. **Data Loading** - Load data with Auto Loader or batch read
4. **Data Quality Assessment** - Check for nulls, duplicates, schema issues
5. **Data Cleaning** - Remove duplicates and handle quality issues
6. **Type Casting & Standardization** - Apply consistent data types and formatting
7. **Schema Enforcement** - Define and validate schemas
8. **Write Operations** - Persist to Delta Lake with enforcement

**Output:** Clean, validated, schema-enforced Silver layer tables ready for Gold layer aggregations.

---
## 1. Configuration
Set up environment parameters and paths for the Silver layer pipeline.

In [0]:
# Configuration parameters - can be overridden via widgets or job parameters
dbutils.widgets.text("environment", "dev", "Environment (dev/staging/prod)")
dbutils.widgets.text("storage_account", "<storage-account>", "Storage Account Name")
dbutils.widgets.text("container", "unity-catalog-storage", "Container Name")

# Get parameter values
env = dbutils.widgets.get("environment")
storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")

# Construct paths
base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{env}/bronze"
silver_base_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{env}/silver"

print(f"Environment: {env}")
print(f"Bronze path: {base_path}")
print(f"Silver path: {silver_base_path}")

---
## 2. Data Discovery
Explore and identify Bronze layer tables to process.

In [0]:
files = [file for file in dbutils.fs.ls(base_path) if 'merged' in file.name]

print(f"Found {len(files)} files/directories in bronze layer with 'merged' in the name:")
for file in files:
    print(f"  - {file.name}")

In [0]:
from pyspark.sql.functions import current_timestamp

print("Loading Bronze layer Delta tables...")
print("="*80)

# Track loaded tables
loaded_tables = {}
errors = []

for file in files:
    table_path = file.path
    table_name = file.name.rstrip('/').replace('.', '_').replace('-', '_')
    
    try:
        # Read Delta table directly (Bronze tables are already Delta format)
        df = spark.read.format("delta").load(table_path)
        
        row_count = df.count()
        col_count = len(df.columns)
        
        # Store DataFrame in globals
        globals()[table_name] = df
        
        loaded_tables[table_name] = {
            'path': table_path,
            'rows': row_count,
            'columns': col_count
        }
        
        print(f"✓ Loaded {table_name}: {row_count:,} rows, {col_count} columns (Delta)")
        display(df.limit(5))
        
    except Exception as e:
        error_msg = f"✗ Failed to load {table_name}: {str(e)}"
        print(error_msg)
        errors.append({'table': table_name, 'error': str(e)})

print(f"\n{'='*80}")
print(f"Summary: Loaded {len(loaded_tables)}/{len(files)} tables successfully")

if errors:
    print(f"Errors: {len(errors)} tables failed to load")
    for err in errors:
        print(f"  - {err['table']}: {err['error']}")
else:
    print("✓ All tables loaded successfully")

---
## 3. Data Quality Assessment
Analyze data quality: check for nulls, duplicates, and schema issues.

---
## 4. Data Cleaning
Remove duplicates and handle data quality issues.

In [0]:
from pyspark.sql.functions import col, count, when, isnan, isnull

def check_data_quality(df, df_name):
    """
    Perform basic data quality checks on a DataFrame
    """
    print(f"\n{'='*80}")
    print(f"Data Quality Report: {df_name}")
    print(f"{'='*80}")
    
    # Basic stats
    total_rows = df.count()
    total_cols = len(df.columns)
    print(f"Total Rows: {total_rows:,}")
    print(f"Total Columns: {total_cols}")
    
    # Check for duplicate column names
    print(f"\nDuplicate Column Analysis:")
    column_names = df.columns
    unique_columns = set(column_names)
    if len(column_names) != len(unique_columns):
        from collections import Counter
        col_counts = Counter(column_names)
        duplicate_cols = {col_name: cnt for col_name, cnt in col_counts.items() if cnt > 1}
        print(f"  ⚠ Duplicate column names found: {len(duplicate_cols)}")
        for col_name, cnt in duplicate_cols.items():
            print(f"    - '{col_name}' appears {cnt} times")
    else:
        print(f"  ✓ No duplicate column names found")
    
    # Check for null values in each column
    print(f"\nNull Value Analysis:")
    null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
    null_summary = null_counts.collect()[0].asDict()
    
    columns_with_nulls = {k: v for k, v in null_summary.items() if v > 0}
    if columns_with_nulls:
        print(f"  Columns with nulls: {len(columns_with_nulls)}")
        for col_name, null_count in sorted(columns_with_nulls.items(), key=lambda x: x[1], reverse=True)[:10]:
            null_pct = (null_count / total_rows) * 100
            print(f"    - {col_name}: {null_count:,} ({null_pct:.2f}%)")
    else:
        print(f"  ✓ No null values found")
    
    # Check for duplicate rows
    distinct_rows = df.distinct().count()
    duplicates = total_rows - distinct_rows
    if duplicates > 0:
        print(f"\nDuplicate Row Analysis:")
        print(f"  ⚠ Duplicate Rows: {duplicates:,} ({(duplicates/total_rows)*100:.2f}%)")
    else:
        print(f"\nDuplicate Row Analysis:")
        print(f"  ✓ No duplicate rows found")
    
    return {
        'name': df_name,
        'total_rows': total_rows,
        'total_cols': total_cols,
        'duplicate_columns': len(column_names) - len(unique_columns),
        'columns_with_nulls': len(columns_with_nulls),
        'duplicate_rows': duplicates
    }

# Run quality checks on merged DataFrames (using correct variable names)
quality_reports = []
for df_name in ['forecast_merged', 'historical_merged', 'global_weather_merged', 'normal_merged']:
    if df_name in globals():
        quality_reports.append(check_data_quality(globals()[df_name], df_name))

print(f"\n{'='*80}")
print(f"Quality Check Summary: {len(quality_reports)} DataFrames checked")
print(f"{'='*80}")

In [0]:
def drop_duplicates(df, df_name):
    """
    Drop duplicate rows from a DataFrame
    """
    print(f"\n{'='*80}")
    print(f"Dropping Duplicates: {df_name}")
    print(f"{'='*80}")
    
    # Get original count
    original_count = df.count()
    
    # Drop duplicates (efficient - single pass)
    df_deduped = df.dropDuplicates()
    
    # Get new count
    new_count = df_deduped.count()
    duplicates_removed = original_count - new_count
    
    if duplicates_removed > 0:
        print(f"  Original rows: {original_count:,}")
        print(f"  Unique rows: {new_count:,}")
        print(f"  ✓ Removed {duplicates_removed:,} duplicate rows ({(duplicates_removed/original_count)*100:.2f}%)")
    else:
        print(f"  ✓ No duplicates found ({original_count:,} rows)")
    
    return df_deduped

# Drop duplicates from merged DataFrames
for df_name in ['forecast_merged', 'historical_merged', 'global_weather_merged', 'normal_merged']:
    if df_name in globals():
        globals()[df_name] = drop_duplicates(globals()[df_name], df_name)

## Type Casting and Standardization

Apply consistent data types and formatting standards across all DataFrames for the Silver layer.

In [0]:
from pyspark.sql.functions import col, upper, initcap, trim, to_date, to_timestamp
from pyspark.sql.types import ByteType, DecimalType

print("\n" + "="*80)
print("Standardizing: forecast_merged")
print("="*80)

# Type casting and standardization
forecast_silver = forecast_merged.select(
    # Standardize text fields - proper case for city names, uppercase for country codes
    initcap(trim(col("city_name"))).alias("city_name"),
    upper(trim(col("country_code"))).alias("country_code"),
    
    # Keep geographic coordinates as double (appropriate precision)
    col("latitude").cast("decimal(9,6)").alias("latitude"),
    col("longitude").cast("decimal(9,6)").alias("longitude"),
    
    # Boolean flags - keep as is
    col("has_ice"),
    col("has_precipitation"),
    col("has_rain"),
    col("has_snow"),
    
    # Probability fields - cast from bigint to tinyint (0-100 range)
    col("ice_probability").cast(ByteType()).alias("ice_probability"),
    col("precipitation_probability").cast(ByteType()).alias("precipitation_probability"),
    col("rain_probability").cast(ByteType()).alias("rain_probability"),
    col("snow_probability").cast(ByteType()).alias("snow_probability"),
    
    # Timestamp - keep as is
    col("processed_timestamp")
)

print(f"✓ Rows: {forecast_silver.count():,}")
print(f"✓ Schema standardized: probability fields (bigint → tinyint), coordinates (double → decimal(9,6))")
print(f"✓ Text standardized: city names (proper case), country codes (uppercase)")
display(forecast_silver.limit(5))

In [0]:
print("\n" + "="*80)
print("Standardizing: historical_merged")
print("="*80)

# Type casting and standardization
historical_silver = historical_merged.select(
    # Standardize text fields
    initcap(trim(col("city_name"))).alias("city_name"),
    upper(trim(col("country_code"))).alias("country_code"),
    
    # Geographic coordinates with consistent precision
    col("latitude").cast("decimal(9,6)").alias("latitude"),
    col("longitude").cast("decimal(9,6)").alias("longitude"),
    
    # Boolean flags
    col("has_precipitation"),
    col("has_snow"),
    
    # Timestamp
    col("processed_timestamp")
)

print(f"✓ Rows: {historical_silver.count():,}")
print(f"✓ Schema standardized: coordinates (double → decimal(9,6))")
print(f"✓ Text standardized: city names (proper case), country codes (uppercase)")
display(historical_silver.limit(5))

In [0]:
print("\n" + "="*80)
print("Standardizing: global_weather_merged")
print("="*80)

# Type casting and standardization
global_weather_silver = global_weather_merged.select(
    # Station ID - uppercase and trimmed
    upper(trim(col("station"))).alias("station"),
    
    # Elevation and coordinates with consistent precision
    col("elevation").cast("decimal(8,2)").alias("elevation"),
    col("latitude").cast("decimal(9,6)").alias("latitude"),
    col("longitude").cast("decimal(9,6)").alias("longitude"),
    
    # Temperature fields - keep as integer (Fahrenheit)
    col("tmax").cast("smallint").alias("tmax"),
    col("tmin").cast("smallint").alias("tmin"),
    
    # Precipitation - standardize precision
    col("prcp").cast("decimal(6,2)").alias("prcp"),
    
    # Timestamp
    col("processed_timestamp")
)

print(f"✓ Rows: {global_weather_silver.count():,}")
print(f"✓ Schema standardized: temperatures (int → smallint), elevation/prcp (double → decimal)")
print(f"✓ Text standardized: station IDs (uppercase)")
display(global_weather_silver.limit(5))

In [0]:
print("\n" + "="*80)
print("Standardizing: normal_merged")
print("="*80)

# Type casting and standardization
normal_silver = normal_merged.select(
    # Station identifiers - uppercase and trimmed
    upper(trim(col("station"))).alias("station"),
    trim(col("station_name")).alias("station_name"),
    
    # Elevation and coordinates with consistent precision
    col("elevation").cast("decimal(8,2)").alias("elevation"),
    col("latitude").cast("decimal(9,6)").alias("latitude"),
    col("longitude").cast("decimal(9,6)").alias("longitude"),
    
    # Date field - convert from string "YYYYMMDD" to proper date type
    to_date(col("date"), "yyyyMMdd").alias("date"),
    
    # Timestamp
    col("processed_timestamp")
)

print(f"✓ Rows: {normal_silver.count():,}")
print(f"✓ Schema standardized: date (string → date), elevation (double → decimal(8,2))")
print(f"✓ Text standardized: station IDs (uppercase), station names (trimmed)")
display(normal_silver.limit(5))

In [0]:
print("\n" + "="*80)
print("STANDARDIZATION SUMMARY")
print("="*80)

standardization_summary = [
    {
        'table': 'forecast_silver',
        'rows': forecast_silver.count(),
        'key_changes': 'Probabilities: bigint→tinyint, Coords: double→decimal(9,6), Text: proper/upper case'
    },
    {
        'table': 'historical_silver',
        'rows': historical_silver.count(),
        'key_changes': 'Coords: double→decimal(9,6), Text: proper/upper case'
    },
    {
        'table': 'global_weather_silver',
        'rows': global_weather_silver.count(),
        'key_changes': 'Temps: int→smallint, Elevation/Prcp: double→decimal, Station: uppercase'
    },
    {
        'table': 'normal_silver',
        'rows': normal_silver.count(),
        'key_changes': 'Date: string→date, Elevation: double→decimal(8,2), Station: uppercase'
    }
]

for summary in standardization_summary:
    print(f"\n{summary['table']}:")
    print(f"  Rows: {summary['rows']:,}")
    print(f"  Changes: {summary['key_changes']}")

print("\n" + "="*80)
print("✓ All DataFrames standardized and ready for Silver layer")
print("="*80)

## Schema Enforcement

Implement strict schema validation and enforcement for Silver layer tables to ensure:
* **Data Type Consistency**: All columns have explicit, validated types
* **Nullability Constraints**: Control which fields can contain null values
* **Schema Evolution Control**: Prevent unexpected schema changes
* **Data Quality**: Reject data that doesn't match expected schema

In [0]:
from pyspark.sql.types import (
    StructType, StructField, StringType, DecimalType, 
    BooleanType, ByteType, ShortType, DateType, TimestampType
)

print("\n" + "="*80)
print("DEFINING SILVER LAYER SCHEMAS")
print("="*80)

# Schema for forecast_silver
forecast_schema = StructType([
    StructField("city_name", StringType(), nullable=False),
    StructField("country_code", StringType(), nullable=False),
    StructField("latitude", DecimalType(9, 6), nullable=False),
    StructField("longitude", DecimalType(9, 6), nullable=False),
    StructField("has_ice", BooleanType(), nullable=False),
    StructField("has_precipitation", BooleanType(), nullable=False),
    StructField("has_rain", BooleanType(), nullable=False),
    StructField("has_snow", BooleanType(), nullable=False),
    StructField("ice_probability", ByteType(), nullable=False),
    StructField("precipitation_probability", ByteType(), nullable=False),
    StructField("rain_probability", ByteType(), nullable=False),
    StructField("snow_probability", ByteType(), nullable=False),
    StructField("processed_timestamp", TimestampType(), nullable=False)
])

# Schema for historical_silver
historical_schema = StructType([
    StructField("city_name", StringType(), nullable=False),
    StructField("country_code", StringType(), nullable=False),
    StructField("latitude", DecimalType(9, 6), nullable=False),
    StructField("longitude", DecimalType(9, 6), nullable=False),
    StructField("has_precipitation", BooleanType(), nullable=False),
    StructField("has_snow", BooleanType(), nullable=False),
    StructField("processed_timestamp", TimestampType(), nullable=False)
])

# Schema for global_weather_silver
global_weather_schema = StructType([
    StructField("station", StringType(), nullable=False),
    StructField("elevation", DecimalType(8, 2), nullable=True),  # Can be null for some stations
    StructField("latitude", DecimalType(9, 6), nullable=False),
    StructField("longitude", DecimalType(9, 6), nullable=False),
    StructField("tmax", ShortType(), nullable=True),  # Temperature can be missing
    StructField("tmin", ShortType(), nullable=True),  # Temperature can be missing
    StructField("prcp", DecimalType(6, 2), nullable=True),  # Precipitation can be missing
    StructField("processed_timestamp", TimestampType(), nullable=False)
])

# Schema for normal_silver
normal_schema = StructType([
    StructField("station", StringType(), nullable=False),
    StructField("station_name", StringType(), nullable=True),  # Name might be missing
    StructField("elevation", DecimalType(8, 2), nullable=True),
    StructField("latitude", DecimalType(9, 6), nullable=False),
    StructField("longitude", DecimalType(9, 6), nullable=False),
    StructField("date", DateType(), nullable=True),  # Date parsing might fail
    StructField("processed_timestamp", TimestampType(), nullable=False)
])

schemas = {
    'forecast_silver': forecast_schema,
    'historical_silver': historical_schema,
    'global_weather_silver': global_weather_schema,
    'normal_silver': normal_schema
}

for table_name, schema in schemas.items():
    print(f"\n✓ {table_name}: {len(schema.fields)} fields defined")
    non_nullable = sum(1 for f in schema.fields if not f.nullable)
    print(f"  - Non-nullable fields: {non_nullable}/{len(schema.fields)}")

print("\n" + "="*80)
print("✓ All schemas defined")
print("="*80)

In [0]:
def validate_schema(df, expected_schema, df_name):
    """
    Validate DataFrame schema against expected schema.
    Returns tuple: (is_valid, validation_messages)
    """
    messages = []
    is_valid = True
    
    print(f"\n{'='*80}")
    print(f"Schema Validation: {df_name}")
    print(f"{'='*80}")
    
    # Check column count
    if len(df.columns) != len(expected_schema.fields):
        is_valid = False
        messages.append(f"✗ Column count mismatch: got {len(df.columns)}, expected {len(expected_schema.fields)}")
    else:
        messages.append(f"✓ Column count: {len(df.columns)}")
    
    # Check each field
    expected_fields = {f.name: f for f in expected_schema.fields}
    actual_fields = {f.name: f for f in df.schema.fields}
    
    # Check for missing columns
    missing_cols = set(expected_fields.keys()) - set(actual_fields.keys())
    if missing_cols:
        is_valid = False
        messages.append(f"✗ Missing columns: {', '.join(missing_cols)}")
    
    # Check for extra columns
    extra_cols = set(actual_fields.keys()) - set(expected_fields.keys())
    if extra_cols:
        is_valid = False
        messages.append(f"✗ Extra columns: {', '.join(extra_cols)}")
    
    # Check data types and nullability for matching columns
    type_mismatches = []
    nullable_mismatches = []
    
    for col_name in expected_fields.keys():
        if col_name in actual_fields:
            expected_field = expected_fields[col_name]
            actual_field = actual_fields[col_name]
            
            # Check data type
            if str(expected_field.dataType) != str(actual_field.dataType):
                type_mismatches.append(
                    f"{col_name}: expected {expected_field.dataType}, got {actual_field.dataType}"
                )
            
            # Check nullability
            if expected_field.nullable != actual_field.nullable:
                nullable_mismatches.append(
                    f"{col_name}: expected nullable={expected_field.nullable}, got nullable={actual_field.nullable}"
                )
    
    if type_mismatches:
        is_valid = False
        messages.append(f"✗ Type mismatches ({len(type_mismatches)}):")
        for mismatch in type_mismatches[:5]:  # Show first 5
            messages.append(f"    - {mismatch}")
    else:
        messages.append("✓ All data types match")
    
    if nullable_mismatches:
        messages.append(f"⚠ Nullability differences ({len(nullable_mismatches)}):")
        for mismatch in nullable_mismatches[:5]:  # Show first 5
            messages.append(f"    - {mismatch}")
    else:
        messages.append("✓ All nullability constraints match")
    
    # Print results
    for msg in messages:
        print(msg)
    
    if is_valid:
        print(f"\n✓ Schema validation PASSED for {df_name}")
    else:
        print(f"\n✗ Schema validation FAILED for {df_name}")
    
    return is_valid, messages

print("✓ Schema validation function defined")

In [0]:
print("\n" + "="*80)
print("VALIDATING ALL SILVER DATAFRAMES")
print("="*80)

validation_results = {}

for df_name, schema in schemas.items():
    if df_name in globals():
        df = globals()[df_name]
        is_valid, messages = validate_schema(df, schema, df_name)
        validation_results[df_name] = {
            'valid': is_valid,
            'messages': messages
        }
    else:
        print(f"\n⚠ {df_name} not found in session")
        validation_results[df_name] = {
            'valid': False,
            'messages': ['DataFrame not found in session']
        }

# Summary
print("\n" + "="*80)
print("VALIDATION SUMMARY")
print("="*80)

passed = sum(1 for r in validation_results.values() if r['valid'])
total = len(validation_results)

print(f"\nPassed: {passed}/{total}")
print(f"Failed: {total - passed}/{total}")

if passed == total:
    print("\n✓ All DataFrames passed schema validation")
else:
    print("\n⚠ Some DataFrames failed validation - review messages above")

---
## 5. Write to Delta Lake
Persist Silver layer tables with schema enforcement enabled.

### Partitioning Strategy

**Why Partition?**
* Improves query performance by pruning irrelevant data
* Optimizes file organization for parallel processing
* Reduces I/O for filtered queries

**Best Practices:**
* Partition on columns frequently used in WHERE clauses
* Target 1GB+ per partition (avoid small files)
* Cardinality: 10-10,000 partitions ideal
* Avoid high-cardinality columns (e.g., timestamps, IDs)

**Recommendations by Table:**

| Table | Rows | Strategy | Rationale |
|-------|------|----------|----------|
| **forecast_silver** | 2,019 | Partition by `country_code` | 43 countries, ~47 rows/partition, good for country-filtered queries |
| **historical_silver** | 92 | No partitioning | Too small - partitioning overhead > benefits |
| **global_weather_silver** | 37 | No partitioning | Too small - single file optimal |
| **normal_silver** | 1,122 | Partition by `year`, `month` | 365 dates, temporal queries common, ~94 rows/month |

In [0]:
from pyspark.sql.functions import year, month

print("\n" + "="*80)
print("APPLYING PARTITIONING STRATEGY")
print("="*80)

# forecast_silver: Add country_code partitioning (already has the column)
print("\n✓ forecast_silver: Will partition by country_code (43 partitions)")
print(f"  Rows: {forecast_silver.count():,}")
print(f"  Avg rows per partition: ~{forecast_silver.count() / 43:.0f}")

# historical_silver: No partitioning needed
print("\n✓ historical_silver: No partitioning (too small - 92 rows)")

# global_weather_silver: No partitioning needed
print("\n✓ global_weather_silver: No partitioning (too small - 37 rows)")

# normal_silver: Add year/month partitioning from date column
print("\n✓ normal_silver: Adding year/month columns for partitioning")
normal_silver_partitioned = normal_silver.withColumn("year", year(col("date"))) \
                                         .withColumn("month", month(col("date")))

print(f"  Rows: {normal_silver_partitioned.count():,}")

# Filter out nulls for partition stats
partition_stats = normal_silver_partitioned.filter(col("date").isNotNull()) \
    .groupBy("year", "month").count().orderBy("year", "month").collect()

if len(partition_stats) > 0:
    print(f"  Partitions: {len(partition_stats)} (year/month combinations)")
    valid_rows = sum(row['count'] for row in partition_stats)
    print(f"  Avg rows per partition: ~{valid_rows / len(partition_stats):.0f}")
    print("\n  Sample partition distribution:")
    for row in partition_stats[:3]:
        print(f"    - {row['year']}-{row['month']:02d}: {row['count']} rows")
else:
    print("  ⚠ No valid dates found for partitioning")

print("\n" + "="*80)
print("✓ Partitioning strategy applied")
print("="*80)

In [0]:
def write_silver_table_partitioned(df, table_name, output_path, partition_cols=None, mode="overwrite"):
    """
    Write DataFrame to Delta table with optional partitioning and schema enforcement.
    
    Args:
        df: DataFrame to write
        table_name: Name of the table
        output_path: Delta table path
        partition_cols: List of column names to partition by (None for no partitioning)
        mode: Write mode (overwrite, append)
    """
    print(f"\n{'='*80}")
    print(f"Writing: {table_name}")
    print(f"{'='*80}")
    print(f"Path: {output_path}")
    print(f"Mode: {mode}")
    print(f"Rows: {df.count():,}")
    
    if partition_cols:
        print(f"Partitioning: {', '.join(partition_cols)}")
    else:
        print("Partitioning: None (single partition)")
    
    try:
        writer = (df.write
            .format("delta")
            .mode(mode)
            .option("mergeSchema", "false")  # Prevent automatic schema evolution
            .option("overwriteSchema", "false" if mode == "append" else "true")
            .option("delta.columnMapping.mode", "name"))
        
        # Apply partitioning if specified
        if partition_cols:
            writer = writer.partitionBy(*partition_cols)
        
        writer.save(output_path)
        
        print(f"✓ Successfully wrote {table_name}")
        
        # Set table properties
        spark.sql(f"""
            ALTER TABLE delta.`{output_path}`
            SET TBLPROPERTIES (
                'delta.minReaderVersion' = '1',
                'delta.minWriterVersion' = '2',
                'delta.columnMapping.mode' = 'name',
                'delta.enableChangeDataFeed' = 'true',
                'delta.autoOptimize.optimizeWrite' = 'true',
                'delta.autoOptimize.autoCompact' = 'true'
            )
        """)
        
        print("✓ Schema enforcement and auto-optimize properties set")
        
        # Show partition info if partitioned
        if partition_cols:
            partition_count = spark.read.format("delta").load(output_path) \
                .select(*partition_cols).distinct().count()
            print(f"✓ Created {partition_count} partitions")
        
        return True
        
    except Exception as e:
        print(f"✗ Failed to write {table_name}: {str(e)}")
        return False

print("\n✓ Partitioned write function defined")

In [0]:
print("\n" + "="*80)
print("WRITING SILVER TABLES WITH PARTITIONING")
print("="*80)

# Define partitioning strategy for each table
tables_config = [
    {
        'df': forecast_silver,
        'name': 'forecast_silver',
        'partition_cols': ['country_code'],
        'reason': 'Partition by country for geographic filtering'
    },
    {
        'df': historical_silver,
        'name': 'historical_silver',
        'partition_cols': None,
        'reason': 'No partitioning - dataset too small (92 rows)'
    },
    {
        'df': global_weather_silver,
        'name': 'global_weather_silver',
        'partition_cols': None,
        'reason': 'No partitioning - dataset too small (37 rows)'
    },
    {
        'df': normal_silver_partitioned,
        'name': 'normal_silver',
        'partition_cols': ['year', 'month'],
        'reason': 'Partition by year/month for temporal queries'
    }
]

write_results = {}

for config in tables_config:
    print(f"\n{'-'*80}")
    print(f"Strategy: {config['reason']}")
    print(f"{'-'*80}")
    
    output_path = f"{silver_base_path}/{config['name']}"
    success = write_silver_table_partitioned(
        df=config['df'],
        table_name=config['name'],
        output_path=output_path,
        partition_cols=config['partition_cols'],
        mode="overwrite"
    )
    write_results[config['name']] = success

# Summary
print("\n" + "="*80)
print("PARTITIONED WRITE SUMMARY")
print("="*80)

success_count = sum(1 for success in write_results.values() if success)
total_count = len(write_results)

print(f"\nSuccessful: {success_count}/{total_count}")
print(f"Failed: {total_count - success_count}/{total_count}")

if success_count == total_count:
    print("\n✓ All silver tables written with optimal partitioning strategy")
    print("\nPartitioning Summary:")
    print("  • forecast_silver: Partitioned by country_code (43 partitions)")
    print("  • historical_silver: No partitioning (too small)")
    print("  • global_weather_silver: No partitioning (too small)")
    print("  • normal_silver: Partitioned by year/month (12 partitions)")
else:
    print("\n⚠ Some tables failed to write - review errors above")

print("\n" + "="*80)